# Churn Prediction ANN — Correct Training Pipeline

This version fixes the preprocessing mismatch. The ColumnTransformer is fit on the original **10 input features** and is saved before any transformed data is reused. The scaler is fit after the transformer. The same two objects must be used in `app.py`.

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import pickle
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [2]:
# Load dataset
df = pd.read_csv(r"D:\ANN\churn-prediction-ann\data\Churn_Modelling.csv")

print("Dataset shape:", df.shape)
print(df.columns.tolist())

Dataset shape: (10000, 14)
['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']


In [11]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# Keep exactly the 10 features used by the application
feature_columns = [
    "CreditScore", "Geography", "Gender", "Age", "Tenure",
    "Balance", "NumOfProducts", "HasCrCard", "IsActiveMember",
    "EstimatedSalary"
]

X = df[feature_columns].copy()
y = df["Exited"].copy()

print("X shape:", X.shape)
print("Features:", X.columns.tolist())
print("y shape:", y.shape)

X shape: (10000, 10)
Features: ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
y shape: (10000,)


In [4]:
# IMPORTANT:
# Fit ColumnTransformer on the ORIGINAL 10-column dataframe.
# Geography and Gender are the only categorical columns.
ct = ColumnTransformer(
    transformers=[
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ["Geography", "Gender"]
        )
    ],
    remainder="passthrough"
)

X_encoded = ct.fit_transform(X)

print("After OneHotEncoder:", X_encoded.shape)
print("Expected: 13 features")

After OneHotEncoder: (10000, 13)
Expected: 13 features


In [5]:
# Fit scaler only on the encoded 13-feature matrix
sc = StandardScaler()
X_scaled = sc.fit_transform(X_encoded)

print("Final X shape:", X_scaled.shape)
print("Final data type:", X_scaled.dtype)

Final X shape: (10000, 13)
Final data type: float64


In [6]:
# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.20,
    random_state=0
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (8000, 13)
X_test: (2000, 13)
y_train: (8000,)
y_test: (2000,)


In [7]:
# Create ANN
ann = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(6, activation="relu"),
    tf.keras.layers.Dense(6, activation="relu"),
    tf.keras.layers.Dense(5, activation="relu"),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

ann.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 6)              │            84 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │            42 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │            35 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │            24 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 190 (760.00 B)

 Trainable params: 190 (760.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
# Train
history = ann.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=15,
    validation_data=(X_test, y_test),
    verbose=1
)

Epoch 1/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.6855 - loss: 0.6073 - val_accuracy: 0.7975 - val_loss: 0.5030
Epoch 2/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.7972 - loss: 0.4738 - val_accuracy: 0.8100 - val_loss: 0.4440
Epoch 3/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8129 - loss: 0.4383 - val_accuracy: 0.8205 - val_loss: 0.4272
Epoch 4/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.8214 - loss: 0.4284 - val_accuracy: 0.8285 - val_loss: 0.4194
Epoch 5/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.8248 - loss: 0.4229 - val_accuracy: 0.8355 - val_loss: 0.4134
Epoch 6/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8236 - loss: 0.4172 - val_accuracy: 0.8335 - val_loss: 0.4093
Epoch 7/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.8265 - loss: 0.4119 - val_accuracy: 0.8315 - val_loss: 0.4078
Epoch 8/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.8251 - loss: 0.4075 - val_accuracy

In [9]:
# Save model + preprocessing objects
model_dir = Path(r"D:\ANN\churn-prediction-ann\models")
model_dir.mkdir(parents=True, exist_ok=True)

ann.save(model_dir / "churn_model.keras")

with open(model_dir / "column_transformer.pkl", "wb") as f:
    pickle.dump(ct, f)

with open(model_dir / "scaler.pkl", "wb") as f:
    pickle.dump(sc, f)

# Also save the exact feature order for the app
with open(model_dir / "feature_columns.pkl", "wb") as f:
    pickle.dump(feature_columns, f)

print("All model files saved successfully!")
print("Saved transformer expects:", ct.n_features_in_, "input features")
print("Saved scaler expects:", sc.n_features_in_, "input features")

All model files saved successfully!
Saved transformer expects: 10 input features
Saved scaler expects: 13 input features


In [10]:
# Final verification: simulate exactly what app.py should do
sample = pd.DataFrame([{
    "CreditScore": 650,
    "Geography": "France",
    "Gender": "Male",
    "Age": 35,
    "Tenure": 5,
    "Balance": 75000.0,
    "NumOfProducts": 1,
    "HasCrCard": 1,
    "IsActiveMember": 1,
    "EstimatedSalary": 100000.0
}])

sample_encoded = ct.transform(sample)
sample_scaled = sc.transform(sample_encoded)
prediction = ann.predict(sample_scaled, verbose=0)[0][0]

print("App input shape:", sample.shape)
print("After transformer:", sample_encoded.shape)
print("After scaler:", sample_scaled.shape)
print("Churn probability:", float(prediction))
print("Prediction:", "Customer will likely CHURN" if prediction >= 0.5 else "Customer will likely STAY")

App input shape: (1, 10)
After transformer: (1, 13)
After scaler: (1, 13)
Churn probability: 0.040204573422670364
Prediction: Customer will likely STAY
